# What is Uproot?

Uproot is a python package which we can use to open. read, analyse and write files in .root format. It isn't the *only* way in which we can open root files in python, but it is one of the most convenient.

In this tutorial, we will use Uproot to open an ePIC simulation file and take a look at what is available.

## What is the .root format?

[ROOT](https://root.cern/) is an analysis framework that is widely used in particle and nuclear physicis. Typically, root files contain information for discrete *events*. An event may represent a crossing of our beams for example. What do we see each time this happens? 

Typically, this information is stored in the form of a *Tree* which contains event by event information as distinct *branches* and *leaves*. You can think of a tree as being a table of information, where the columns of the table are our branches. Each row is a new event.

### Branches and Leaves

A leaf is the simplest component of a ROOT tree and is typically a single piece of data or information for an event. For example, the energy readout of one element of a detector (as a floating point number) would be a common leaf. A branch is a collection of such objects, so continuing our example, this could be a collection of values/information from one detetor.

Note that our branch could also be an array, for example, an array of all energy values for particles in an event.

- **Note that the size of such an array may not be the same for every event!**
    - These are "jagged arrays" 
    - We could have 3 particles produced in one bunch crossing, 0 the next, 4 the time after that for example
    - ROOT handles this without issue, in uproot, these can be handled quite straightforwardly too
    - Handling jagged arrays with other python packages can be problematic

## Event by event

The key thing here is that the information is all stored in an event by event manner. We can interate through our tree, checking the values of different components for each distinct event recorded. 

This is slightly abstract to talk about, so let's see it in action.

# Setup

In [ ]:
#Import some packages we'll need, specifically, uproot
import uproot as up
import os
from XRootD import client
import awkward as ak
import numpy as np
#import pandas as pd
#import scipy
import matplotlib as mpl

# Opening Files and Browsing Info

In [ ]:
 # The file we downloaded previously, change as desired
fname = "/Users/hszumila/Desktop/Day_1_Tutorial_Input.root"
if os.path.isfile(fname):
    file=up.open(fname)
    file
else:
    print("Error opening file - ", fname, " check your fname variable!")

So long as our file exists, we should now have it open and assigned to the variable "file". We can now take a look at what it contains.

In [ ]:
file.keys()

We can see that our file contains three "keys", what are these? Let's check -

In [ ]:
file.classnames()

Ok, they're all trees (good!), we will choose the "events" tree and take a look at what it contains -

In [ ]:
tree = file['events'] # Assign our tree of choice to a variable
tree # Take a look at it

We can use the .keys() function again to see what our 806 branches are -

In [ ]:
tree.keys(recursive=False)

That's a lot of branches! Also, this is just a list of the branch names, not their actual values, or even their branch elements. We'll get to that in a moment. Before that, let's see if we can make sense of the branch names. Most branches follow a similar format -

**B0ECalRecHits** - What is this? Let's break it down a bit -

- B0ECal
    - This first part is telling us that this branch relates to the *B0* detector, specifically the *E*lectromagnetic *Cal*orimeter
- Rec
    - This is telling us that this branch contains *rec*onstructed infromation (as opposed to raw or MC information)
- Hits
      - This is saying we have *hit* information, as opposed to tracks or clusters

## Exercise

- Pick another branch name from the list and see if you can determine the detector system it belongs to
    - What type of detector is it? Tracker? Calorimeter? PID?

Our huge list is still a bit excessive though. Luckily, we can filter it quite simply by another addition to our .keys call -

In [ ]:
tree.keys(filter_name="*cal*",recursive=False)

## Exercise

- Find all branches that correspond to information about the end cap electromagnetic calorimeter.
    - Identify the relevant branch naming scheme for this detector based upon the branches you've identified
    - **Hint, remember that the * is a wildcard character, try to find strings that match this detector**

In [ ]:
tree.keys(filter_name="*cal*",recursive=False)

# Getting Information from Branches

Ok, we can now hopefully find branches in our file, but what do they actually *contain* and how can we access that? Let's pick one of our branches above and see what it contains - 

In [ ]:
branches = tree["B0ECalClusters"]
branches.keys()

So our branch actually has a whole bunch of sub elements. Let's pick one of these and and convert it to an array -

In [ ]:
branch = branches["B0ECalClusters.energy"].array()
print(branch)
print(branch[23])
print(len(branch))

So, we converted the branch B0ECalClusters.energy to an array and printed it. We then printed event 23 specifically. We also then saw that there are 410 elements in our branch. This should correspond to the number of entries in our tree, so let's check that -

In [ ]:
print(tree.num_entries)

Ok, great. Things are starting to make sense!

Note that we can also just convert all of the branches to arrays in one go, skipping a step. We can single out specific entries/branches in a few ways as well - 

In [ ]:
branches = tree["B0ECalClusters"].arrays()
branches[0]

In [ ]:
# We can also convert our entry in all branch elements to a list and print that. In doing so, we won't lose stuff to our elipses again
print(branches[0].tolist())

In [ ]:
branches['B0ECalClusters.position.x'][0]

In [ ]:
branch = branches['B0ECalClusters.energy']
for i in range (0,10):
    print(branch[i])

## Exercise 

1. Find a branch of our tree which contains reconstructed charged particles.
2. Identify all of the quantities stored for these particles in an event.
3. For the first 100 events, print the number of reconstructed charged particles found within each event.

# Filtering Branches with Masks/Cuts

Finally, as a teaser for what we'll be doing a bit later. We can also impose conditions on our branches -

In [ ]:
branches = tree["ReconstructedChargedParticles"].arrays()
branches['ReconstructedChargedParticles.energy'] > 1

In [ ]:
Energy_Selection = branches['ReconstructedChargedParticles.energy'] > 1
branches['ReconstructedChargedParticles.energy'][Energy_Selection]

In [ ]:
for i in range (0,10):
    print(len(branches['ReconstructedChargedParticles.energy'][i]), len(branches['ReconstructedChargedParticles.energy'][Energy_Selection][i]))

Note that we can apply our selection to **other** branches as well -

In [ ]:
for i in range (0,10):
    print(len(branches['ReconstructedChargedParticles.momentum.z'][i]), len(branches['ReconstructedChargedParticles.momentum.z'][Energy_Selection][i]))

## Exercise 

1. Find the number of reconstructed positively charged particles and the number of reconstructed negatively charged particles.
2. Verify that the sum of these two numbers is equal to the **total** number of reconstructed charged particles.
3. Find the total energy of all negatively charged reconstructed particles.
4. Find the number of negatively charged particles with an energy greater than 5. Verify this in two ways.

**Hint** - We can use sum, np.sum, len and other methods to manipulate arrays to get a lot of this information